# Week 6 — Advanced Retrieval: Transform and Measure

This notebook extends the Week 5 retrieval pipeline using the same 16 chunks and the same 10 frozen evaluation queries.

1. Recreate the Week 5 baseline retriever.
2. Apply a Gemini multi-query transformation.
3. Measure transformation-only retrieval.
4. Re-rank transformed candidates with a local cross-encoder.
5. Compare baseline, transformed, and final results per query.
6. Identify which query types benefited and which did not.
7. Find a transformation regression, or justify a no-regression result.

The optional schema-constrained extraction enrichment is not included.


## 1. Setup

Runs on CPU in Google Colab or local JupyterLab. Credential lookup order: Colab Secrets, environment variable, then local `.env`.

Gemini returns two rewrites in one call per query. A cache entry is valid only when it contains **two unique, non-empty rewrites**.


In [56]:
import sys, subprocess, importlib.util
mods={"numpy":"numpy","pandas":"pandas","sentence_transformers":"sentence-transformers","faiss":"faiss-cpu","google.genai":"google-genai","dotenv":"python-dotenv"}
need=[p for m,p in mods.items() if importlib.util.find_spec(m) is None]
if need:
    subprocess.check_call([sys.executable,"-m","pip","install",*need])


In [57]:
import os, json, time, math, re
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd, faiss
from dotenv import load_dotenv, find_dotenv
from sentence_transformers import SentenceTransformer, CrossEncoder
from IPython.display import display, Markdown
TOP_K=3; CANDIDATE_K=8
EMBED_MODEL="sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL="cross-encoder/ms-marco-MiniLM-L-6-v2"
GEMINI_MODEL="gemini-3.5-flash-lite"
CACHE_PATH=Path("week6_query_cache.json")


## 2. Week 5 corpus and frozen evaluation set

The corpus and gold labels are unchanged from Week 5 so the comparison remains valid.


In [58]:
DOCUMENTS=[
("D01","Request intake",[("S1","Required fields","A new operations request must include a short title, requester name, business team, problem statement, and desired outcome. The requester can attach example files but attachments are optional."),("S2","Duplicate requests","Before opening a new request, search the request list by title and team. If an existing request covers the same problem, add a comment to that request rather than creating a duplicate.")]),
("D02","Triage and priority",[("S1","Priority definitions","Priority 1 means a production service is unavailable to multiple people. Priority 2 means a blocked business process with a manual workaround. Priority 3 means an improvement or a question without an active outage."),("S2","First review","The operations coordinator checks completeness and assigns an owner within two business days. Missing intake fields are sent back to the requester before priority is assigned.")]),
("D03","Approval workflow",[("S1","Approval conditions","A request that changes customer-facing content or grants access to protected data requires manager approval before implementation. An internal wording correction with no data access change does not require approval."),("S2","Rejected approval","When a manager rejects a request, record the reason and set approval result to Rejected. The owner may revise the request and seek a new approval; rejection does not mean it is complete.")]),
("D04","Delivery and review",[("S1","Human review","Generated drafts must be reviewed for factual correctness by the assigned owner. A workflow must not automatically send a draft to customers or publish it internally before a person approves the final text."),("S2","Completion","Set status to Completed only after the requester accepts the delivered result and the owner records the outcome. Saving a generated draft alone does not close a request.")]),
("D05","SharePoint file handling",[("S1","Source files","Store source files in the team SharePoint document library. In the request, link the specific file and record its version; do not paste the full file into the request description."),("S2","Access permissions","Requesters may see their own request and shared team files. An owner checks a recipient's access before sharing a source link; a link does not grant permission by itself.")]),
("D06","Automation monitoring",[("S1","Flow failure","If a Power Automate run fails, inspect its run history and note the failed action and error in the request. Retry only after correcting the cause, to avoid sending duplicate notifications."),("S2","Success checks","For a new automation, test one ordinary case and one missing-field case. Record expected versus observed results and ask the owner to sign off before activation.")]),
("D07","Data safeguards",[("S1","Sensitive content","Remove customer identifiers and confidential attachments from example prompts unless their use is authorized. Use a small, sanitized excerpt when a full document is unnecessary."),("S2","Retention","Keep the request's decision and final outcome in the request list. Delete temporary test exports when validation finishes according to the team's retention schedule.")]),
("D08","Notifications and escalation",[("S1","Status updates","The owner posts a status update on the request when work starts and when the expected delivery date changes. Routine updates are visible to the requester in the list."),("S2","Escalation","Escalate a production outage to the on-call owner immediately and record the escalation time. A non-outage item waiting more than two business days for assignment goes to the operations coordinator.")])]
CHUNKS=[{"chunk_id":f"{d}-{s}","title":t,"heading":h,"body":b,"text":f"{t} — {h}. {b}"} for d,t,secs in DOCUMENTS for s,h,b in secs]
by_id={c["chunk_id"]:c for c in CHUNKS}
TESTS=[
("Q01","What fields must a requester provide when opening a request?",{"D01-S1"}),
("Q02","What should I do if someone has already reported the same problem?",{"D01-S2"}),
("Q03","Which priority applies when a process is blocked but a workaround exists?",{"D02-S1"}),
("Q04","How soon is an owner assigned, and what happens to an incomplete request?",{"D02-S2"}),
("Q05","When does a request require manager approval?",{"D03-S1"}),
("Q06","What happens after a manager rejects a request?",{"D03-S2"}),
("Q07","Can an AI draft be published or emailed without a person checking it?",{"D04-S1"}),
("Q08","When can the owner mark the request Completed?",{"D04-S2"}),
("Q09","How do I investigate a failed Power Automate flow before retrying?",{"D06-S1"}),
("Q10","How quickly should a production outage be escalated?",{"D08-S2"})]
display(pd.DataFrame([{"qid":q,"query":x,"gold":next(iter(g))} for q,x,g in TESTS]))


,qid,query,gold
0,Q01,What fields must a requester provide when open...,D01-S1
1,Q02,What should I do if someone has already report...,D01-S2
2,Q03,Which priority applies when a process is block...,D02-S1
3,Q04,"How soon is an owner assigned, and what happen...",D02-S2
4,Q05,When does a request require manager approval?,D03-S1
5,Q06,What happens after a manager rejects a request?,D03-S2
6,Q07,Can an AI draft be published or emailed withou...,D04-S1
7,Q08,When can the owner mark the request Completed?,D04-S2
8,Q09,How do I investigate a failed Power Automate f...,D06-S1
9,Q10,How quickly should a production outage be esca...,D08-S2


## 3. Week 5 baseline retriever


In [59]:
embedder=SentenceTransformer(EMBED_MODEL)
texts=[c["text"] for c in CHUNKS]
V=embedder.encode(texts,normalize_embeddings=True,show_progress_bar=False).astype("float32")
index=faiss.IndexFlatIP(V.shape[1]); index.add(V)
def retrieve(q,k=TOP_K):
    v=embedder.encode([q],normalize_embeddings=True,show_progress_bar=False).astype("float32")
    scores,idx=index.search(v,k)
    return [{"chunk_id":CHUNKS[j]["chunk_id"],"score":float(sc),"rank":r+1} for r,(j,sc) in enumerate(zip(idx[0],scores[0]))]
baseline={qid:retrieve(q) for qid,q,_ in TESTS}
display(pd.DataFrame([{"qid":qid,"baseline":[x["chunk_id"] for x in baseline[qid]]} for qid,_,_ in TESTS]))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,qid,baseline
0,Q01,"[D01-S1, D01-S2, D05-S2]"
1,Q02,"[D01-S2, D06-S1, D08-S2]"
2,Q03,"[D02-S1, D02-S2, D03-S2]"
3,Q04,"[D02-S2, D08-S1, D04-S2]"
4,Q05,"[D03-S2, D03-S1, D02-S2]"
5,Q06,"[D03-S2, D04-S2, D08-S1]"
6,Q07,"[D04-S1, D04-S2, D06-S2]"
7,Q08,"[D08-S1, D04-S2, D02-S2]"
8,Q09,"[D06-S1, D06-S2, D08-S2]"
9,Q10,"[D08-S2, D02-S2, D08-S1]"


## 4. Query transformation

Gemini produces two alternate search queries that preserve the original intent but use likely operations-policy vocabulary. The original query is retained during fusion. Cached rewrites are de-duplicated; any query with fewer than two unique rewrites is regenerated.


In [60]:
def load_gemini_key():
    try:
        from google.colab import userdata
        for name in ("GEMINI_API_KEY","GEMENI_API_KEY"):
            try:
                value=userdata.get(name)
                if value: return value.strip(), f"Colab Secrets ({name})"
            except Exception: pass
    except ImportError: pass
    for name in ("GEMINI_API_KEY","GEMENI_API_KEY"):
        value=os.getenv(name)
        if value: return value.strip(), f"environment variable ({name})"
    env_file=find_dotenv(usecwd=True)
    if env_file:
        load_dotenv(env_file,override=False)
        for name in ("GEMINI_API_KEY","GEMENI_API_KEY"):
            value=os.getenv(name)
            if value: return value.strip(), f".env ({name})"
    return None,None

def unique_rewrites(values):
    out=[]
    for value in values or []:
        if not isinstance(value,str): continue
        value=value.strip().strip('"')
        if value and value not in out: out.append(value)
    return out

cache=json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
cache={qid:unique_rewrites(values)[:2] for qid,values in cache.items()}
key,key_source=load_gemini_key()
needs_live_calls=any(len(cache.get(qid,[]))<2 for qid,_,_ in TESTS)
if key_source:
    print(f"Gemini API key loaded from {key_source}.")
elif needs_live_calls:
    raise RuntimeError("Gemini API key not found. Add GEMINI_API_KEY to Colab Secrets, the environment, or a local .env file.")
else:
    print("Complete unique-rewrite cache found; Gemini API key is not required for this run.")

if key and needs_live_calls:
    from google import genai
    client=genai.Client(api_key=key)
    def generate_rewrites(query,max_retries=4):
        prompt=("Rewrite the semantic-search query below two different ways using likely operations-policy vocabulary. "
                "Preserve the original intent exactly. Return exactly two concise queries, one per line, prefixed 1. and 2.\n\n"
                f"Original query: {query}")
        for _ in range(max_retries):
            try:
                response=client.models.generate_content(model=GEMINI_MODEL,contents=prompt)
                lines=[re.sub(r"^\\s*\\d+[.):-]\\s*","",line).strip() for line in response.text.splitlines() if line.strip()]
                rewrites=unique_rewrites(lines)
                if len(rewrites)>=2: return rewrites[:2]
                raise ValueError(f"Gemini returned fewer than two unique rewrites: {response.text!r}")
            except Exception as exc:
                msg=str(exc)
                if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                    m=re.search(r"retry in ([0-9.]+)s",msg,re.I)
                    delay=float(m.group(1))+2 if m else 40
                    print(f"Rate limit reached. Retrying in {delay:.0f} seconds...")
                    time.sleep(delay); continue
                raise
        raise RuntimeError("Gemini did not return two unique rewrites after retries.")

    for qid,query,_ in TESTS:
        existing=unique_rewrites(cache.get(qid,[]))
        if len(existing)>=2:
            cache[qid]=existing[:2]; continue
        print(f"Generating unique rewrites for {qid}...")
        cache[qid]=generate_rewrites(query)
        CACHE_PATH.write_text(json.dumps(cache,indent=2))
        time.sleep(2)

transformed={qid:unique_rewrites(cache[qid])[:2] for qid,_,_ in TESTS}
assert all(len(v)==2 and len(set(v))==2 for v in transformed.values()), "Each query must have two unique rewrites."
CACHE_PATH.write_text(json.dumps(transformed,indent=2))
display(pd.DataFrame([{"qid":qid,"original":q,"rewrite_1":transformed[qid][0],"rewrite_2":transformed[qid][1]} for qid,q,_ in TESTS]))


Gemini API key loaded from Colab Secrets (GEMINI_API_KEY).
Generating unique rewrites for Q04...


,qid,original,rewrite_1,rewrite_2
0,Q01,What fields must a requester provide when open...,Mandatory ticket creation fields and required ...,Mandatory input fields for request creation
1,Q02,What should I do if someone has already report...,How to handle duplicate incident reports or ex...,Incident management duplicate reporting procedure
2,Q03,Which priority applies when a process is block...,What is the precedence rule for a blocked proc...,process blocking workaround priority conflict ...
3,Q04,"How soon is an owner assigned, and what happen...",1. What is the SLA for owner assignment and th...,2. What are the timelines for routing ownershi...
4,Q05,When does a request require manager approval?,conditions requiring management authorization ...,Conditions for mandatory manager approval on r...
5,Q06,What happens after a manager rejects a request?,What is the standard operating procedure follo...,manager rejection workflow post-decision proce...
6,Q07,Can an AI draft be published or emailed withou...,What are the compliance protocols for automate...,What are the mandatory human-in-the-loop revie...
7,Q08,When can the owner mark the request Completed?,When is a request status eligible for transiti...,When is the request eligible for owner status ...
8,Q09,How do I investigate a failed Power Automate f...,What is the standard troubleshooting procedure...,What is the standard operating procedure for d...
9,Q10,How quickly should a production outage be esca...,SLA incident escalation timeline for critical ...,What is the required escalation SLA for a prod...


## 5. Transformation-only retrieval

This stage measures query transformation before cross-encoder re-ranking.


In [61]:
def rrf(lists,k=60):
    scores=defaultdict(float)
    for ranked in lists:
        for item in ranked: scores[item["chunk_id"]]+=1/(k+item["rank"])
    return [cid for cid,_ in sorted(scores.items(),key=lambda z:z[1],reverse=True)]
transformed_candidates={qid:rrf([retrieve(q,CANDIDATE_K)]+[retrieve(rw,CANDIDATE_K) for rw in transformed[qid]])[:CANDIDATE_K] for qid,q,_ in TESTS}
transform_only={qid:candidates[:TOP_K] for qid,candidates in transformed_candidates.items()}
display(pd.DataFrame([{"qid":qid,"transform_only_ids":transform_only[qid]} for qid,_,_ in TESTS]))


,qid,transform_only_ids
0,Q01,"[D01-S1, D01-S2, D04-S2]"
1,Q02,"[D01-S2, D08-S2, D04-S2]"
2,Q03,"[D02-S1, D02-S2, D03-S2]"
3,Q04,"[D08-S1, D02-S2, D04-S2]"
4,Q05,"[D03-S2, D03-S1, D01-S1]"
5,Q06,"[D03-S2, D04-S2, D03-S1]"
6,Q07,"[D04-S1, D04-S2, D06-S2]"
7,Q08,"[D08-S1, D04-S2, D02-S2]"
8,Q09,"[D06-S1, D06-S2, D02-S2]"
9,Q10,"[D08-S2, D02-S2, D08-S1]"


## 6. Cross-encoder re-ranking

The cross-encoder jointly scores each candidate with the original user query. This adds CPU latency but can improve ordering among semantically similar candidates.


In [62]:
reranker=CrossEncoder(RERANK_MODEL)
def rerank(q,candidates):
    scores=reranker.predict([(q,by_id[c]["text"]) for c in candidates])
    return [c for c,_ in sorted(zip(candidates,scores),key=lambda z:float(z[1]),reverse=True)[:TOP_K]]
finals={qid:rerank(q,transformed_candidates[qid]) for qid,q,_ in TESTS}
display(pd.DataFrame([{"qid":qid,"final_ids":finals[qid]} for qid,_,_ in TESTS]))


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,qid,final_ids
0,Q01,"[D01-S1, D01-S2, D02-S2]"
1,Q02,"[D01-S2, D06-S1, D08-S2]"
2,Q03,"[D02-S1, D02-S2, D03-S1]"
3,Q04,"[D02-S2, D08-S2, D08-S1]"
4,Q05,"[D03-S1, D03-S2, D01-S1]"
5,Q06,"[D03-S2, D04-S2, D03-S1]"
6,Q07,"[D04-S1, D04-S2, D06-S2]"
7,Q08,"[D04-S2, D08-S1, D02-S2]"
8,Q09,"[D06-S1, D06-S2, D02-S2]"
9,Q10,"[D08-S2, D02-S1, D02-S2]"


## 7. Before-and-after measurement


In [63]:
def metrics(ids,gold):
    hits=[int(x in gold) for x in ids[:TOP_K]]
    rr=next((1/i for i,x in enumerate(ids[:TOP_K],1) if x in gold),0)
    dcg=sum(rel/math.log2(i+2) for i,rel in enumerate(hits))
    return {"p3":sum(hits)/TOP_K,"r3":sum(hits)/len(gold),"hit1":hits[0],"mrr":rr,"ndcg3":dcg}
rows=[]
for qid,q,gold in TESTS:
    b=[x["chunk_id"] for x in baseline[qid]]; t=transform_only[qid]; f=finals[qid]
    bm,tm,fm=metrics(b,gold),metrics(t,gold),metrics(f,gold)
    row={"qid":qid,"query":q,"gold":next(iter(gold)),"baseline_ids":b,"transform_ids":t,"final_ids":f}
    row.update({f"base_{k}":v for k,v in bm.items()}); row.update({f"transform_{k}":v for k,v in tm.items()}); row.update({f"final_{k}":v for k,v in fm.items()})
    row.update({f"delta_transform_{k}":tm[k]-bm[k] for k in bm}); row.update({f"delta_final_{k}":fm[k]-bm[k] for k in bm}); rows.append(row)
results=pd.DataFrame(rows)
display(results[["qid","query","gold","baseline_ids","transform_ids","final_ids","base_hit1","transform_hit1","final_hit1","delta_transform_mrr","delta_final_mrr"]])
metric_names=["p3","r3","hit1","mrr","ndcg3"]
summary=pd.DataFrame({"baseline":[results[f"base_{m}"].mean() for m in metric_names],"transform_only":[results[f"transform_{m}"].mean() for m in metric_names],"final_reranked":[results[f"final_{m}"].mean() for m in metric_names]},index=metric_names)
summary["transform_delta"]=summary.transform_only-summary.baseline; summary["final_delta"]=summary.final_reranked-summary.baseline
display(summary.round(3))


,qid,query,gold,baseline_ids,transform_ids,final_ids,base_hit1,transform_hit1,final_hit1,delta_transform_mrr,delta_final_mrr
0,Q01,What fields must a requester provide when open...,D01-S1,"[D01-S1, D01-S2, D05-S2]","[D01-S1, D01-S2, D04-S2]","[D01-S1, D01-S2, D02-S2]",1,1,1,0.0,0.0
1,Q02,What should I do if someone has already report...,D01-S2,"[D01-S2, D06-S1, D08-S2]","[D01-S2, D08-S2, D04-S2]","[D01-S2, D06-S1, D08-S2]",1,1,1,0.0,0.0
2,Q03,Which priority applies when a process is block...,D02-S1,"[D02-S1, D02-S2, D03-S2]","[D02-S1, D02-S2, D03-S2]","[D02-S1, D02-S2, D03-S1]",1,1,1,0.0,0.0
3,Q04,"How soon is an owner assigned, and what happen...",D02-S2,"[D02-S2, D08-S1, D04-S2]","[D08-S1, D02-S2, D04-S2]","[D02-S2, D08-S2, D08-S1]",1,0,1,-0.5,0.0
4,Q05,When does a request require manager approval?,D03-S1,"[D03-S2, D03-S1, D02-S2]","[D03-S2, D03-S1, D01-S1]","[D03-S1, D03-S2, D01-S1]",0,0,1,0.0,0.5
5,Q06,What happens after a manager rejects a request?,D03-S2,"[D03-S2, D04-S2, D08-S1]","[D03-S2, D04-S2, D03-S1]","[D03-S2, D04-S2, D03-S1]",1,1,1,0.0,0.0
6,Q07,Can an AI draft be published or emailed withou...,D04-S1,"[D04-S1, D04-S2, D06-S2]","[D04-S1, D04-S2, D06-S2]","[D04-S1, D04-S2, D06-S2]",1,1,1,0.0,0.0
7,Q08,When can the owner mark the request Completed?,D04-S2,"[D08-S1, D04-S2, D02-S2]","[D08-S1, D04-S2, D02-S2]","[D04-S2, D08-S1, D02-S2]",0,0,1,0.0,0.5
8,Q09,How do I investigate a failed Power Automate f...,D06-S1,"[D06-S1, D06-S2, D08-S2]","[D06-S1, D06-S2, D02-S2]","[D06-S1, D06-S2, D02-S2]",1,1,1,0.0,0.0
9,Q10,How quickly should a production outage be esca...,D08-S2,"[D08-S2, D02-S2, D08-S1]","[D08-S2, D02-S2, D08-S1]","[D08-S2, D02-S1, D02-S2]",1,1,1,0.0,0.0


,baseline,transform_only,final_reranked,transform_delta,final_delta
p3,0.333,0.333,0.333,0.000,0.000
r3,1.000,1.000,1.000,0.000,0.000
hit1,0.800,0.700,1.000,-0.100,0.200
mrr,0.900,0.850,1.000,-0.050,0.100
ndcg3,0.926,0.889,1.000,-0.037,0.074


## 8. Which query types benefited?

These are retrieval-behavior query types: gold already rank 1 vs. gold not rank 1 at baseline.


In [64]:
analysis_rows=[]
for _,row in results.iterrows():
    query_type="baseline gold not rank 1" if row["base_hit1"]==0 else "baseline already rank 1"
    analysis_rows.append({"qid":row.qid,"query_type":query_type,"transform_mrr_delta":row.delta_transform_mrr,"final_mrr_delta":row.delta_final_mrr,"benefited_transform":row.delta_transform_mrr>0,"benefited_final":row.delta_final_mrr>0})
query_type_results=pd.DataFrame(analysis_rows); display(query_type_results)
grouped=query_type_results.groupby("query_type",observed=True).agg(queries=("qid","count"),mean_transform_mrr_delta=("transform_mrr_delta","mean"),mean_final_mrr_delta=("final_mrr_delta","mean"),transform_benefits=("benefited_transform","sum"),final_benefits=("benefited_final","sum"))
display(grouped.round(3))


,qid,query_type,transform_mrr_delta,final_mrr_delta,benefited_transform,benefited_final
0,Q01,baseline already rank 1,0.0,0.0,False,False
1,Q02,baseline already rank 1,0.0,0.0,False,False
2,Q03,baseline already rank 1,0.0,0.0,False,False
3,Q04,baseline already rank 1,-0.5,0.0,False,False
4,Q05,baseline gold not rank 1,0.0,0.5,False,True
5,Q06,baseline already rank 1,0.0,0.0,False,False
6,Q07,baseline already rank 1,0.0,0.0,False,False
7,Q08,baseline gold not rank 1,0.0,0.5,False,True
8,Q09,baseline already rank 1,0.0,0.0,False,False
9,Q10,baseline already rank 1,0.0,0.0,False,False


,queries,mean_transform_mrr_delta,mean_final_mrr_delta,transform_benefits,final_benefits
query_type,,,,,
baseline already rank 1,8,-0.062,0.0,0,0
baseline gold not rank 1,2,0.000,0.5,0,2


## 9. Required failure case: transformation regression

This section inspects transformation-only retrieval so a later reranker cannot hide a regression.


In [65]:
transform_regressions=results[results["delta_transform_mrr"]<0].sort_values("delta_transform_mrr")
if len(transform_regressions):
    worst=transform_regressions.iloc[0]; qid=worst.qid
    display(Markdown(f"### Transformation regression: {qid}\n**Original query:** {worst['query']}\n\n**Rewrites:** {transformed[qid]}\n\n**Gold chunk:** {worst['gold']} — {by_id[worst['gold']]['text']}\n\n**Baseline ranking:** {worst['baseline_ids']}\n\n**Transformation-only ranking:** {worst['transform_ids']}\n\n**Final reranked ranking:** {worst['final_ids']}\n\n**Explanation:** The transformed wording appears to have emphasized a related operations concept strongly enough to move a less-direct chunk above the gold chunk. This is an inference from the changed ranking, not proof of the embedding model's internal cause. The reranker is shown separately so a repaired ranking does not hide the transformation regression."))
else:
    display(Markdown("### No transformation regression observed\nNo query had lower MRR after transformation-only RRF than under the Week 5 baseline. Across this set, the rewrites preserved the core requested action/entity and the original query remained part of fusion, reducing the chance that a rewrite displaced the correct chunk. This is an interpretation of the observed rankings, not a universal guarantee."))


### Transformation regression: Q04
**Original query:** How soon is an owner assigned, and what happens to an incomplete request?

**Rewrites:** ['1. What is the SLA for owner assignment and the disposition protocol for abandoned or pending requests?', '2. What are the timelines for routing ownership and the workflow for unassigned or incomplete tickets?']

**Gold chunk:** D02-S2 — Triage and priority — First review. The operations coordinator checks completeness and assigns an owner within two business days. Missing intake fields are sent back to the requester before priority is assigned.

**Baseline ranking:** ['D02-S2', 'D08-S1', 'D04-S2']

**Transformation-only ranking:** ['D08-S1', 'D02-S2', 'D04-S2']

**Final reranked ranking:** ['D02-S2', 'D08-S2', 'D08-S1']

**Explanation:** The transformed wording appears to have emphasized a related operations concept strongly enough to move a less-direct chunk above the gold chunk. This is an inference from the changed ranking, not proof of the embedding model's internal cause. The reranker is shown separately so a repaired ranking does not hide the transformation regression.

## 10. Measured interpretation


In [66]:
b=summary.baseline; t=summary.transform_only; f=summary.final_reranked
display(Markdown(f"### Measured interpretation\n- **Baseline:** P@3 {b.p3:.3f}, R@3 {b.r3:.3f}, Hit@1 {b.hit1:.3f}, MRR {b.mrr:.3f}, nDCG@3 {b.ndcg3:.3f}.\n- **Transformation-only:** P@3 {t.p3:.3f}, R@3 {t.r3:.3f}, Hit@1 {t.hit1:.3f}, MRR {t.mrr:.3f}, nDCG@3 {t.ndcg3:.3f}.\n- **After cross-encoder reranking:** P@3 {f.p3:.3f}, R@3 {f.r3:.3f}, Hit@1 {f.hit1:.3f}, MRR {f.mrr:.3f}, nDCG@3 {f.ndcg3:.3f}.\n\nTransformation MRR delta vs. baseline: **{summary.loc['mrr','transform_delta']:+.3f}**. Final MRR delta vs. baseline: **{summary.loc['mrr','final_delta']:+.3f}**.\n\nThe cross-encoder adds CPU latency because it jointly scores every query-candidate pair. That cost is justified when first-stage retrieval returns several plausible chunks and ranking quality matters. Cached rewrites remove repeated Gemini calls on later runs."))


### Measured interpretation
- **Baseline:** P@3 0.333, R@3 1.000, Hit@1 0.800, MRR 0.900, nDCG@3 0.926.
- **Transformation-only:** P@3 0.333, R@3 1.000, Hit@1 0.700, MRR 0.850, nDCG@3 0.889.
- **After cross-encoder reranking:** P@3 0.333, R@3 1.000, Hit@1 1.000, MRR 1.000, nDCG@3 1.000.

Transformation MRR delta vs. baseline: **-0.050**. Final MRR delta vs. baseline: **+0.100**.

The cross-encoder adds CPU latency because it jointly scores every query-candidate pair. That cost is justified when first-stage retrieval returns several plausible chunks and ranking quality matters. Cached rewrites remove repeated Gemini calls on later runs.

## AI use

ChatGPT was used to debug the Week 6 notebook, preserve the Week 5 evaluation set, and grammatically check/improve documentation. Gemini Flash-Lite is used for the measured query transformations. The analysis, measurements, and final decisions are based on the executed notebook results.
